# Exercise 8: Hyperpameter tuning/optimization: Random and Grid Search

Pima Indian Diabetes dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

The datasets consists of several medical predictor variables and one target variable, _Outcome_. Predictor variables includes the number of pregnancies the patient has had, their BMI, insulin level, age, and so on.

[Pima Indian Diabes dataset in Kaggle](https://www.kaggle.com/uciml/pima-indians-diabetes-database)

Dataset can found from Moodle -> Dataset.

__This exercises HAVE TO be done in CSC Puhti cloud environment.__

__NOTE: Try to get as good as possible accuracy. Accurary is one of the evalution criteria__.

TOP results will be found from [here](https://student.labranet.jamk.fi/~rmika/DLNN/TOP-scores.htm)



## Grid Search

Lots of code examples can be found from [this tutorial](https://machinelearningmastery.com/grid-search-hyperparameters-deep-learning-models-python-keras/) for different hyperparameters.

### Use scikit-learn to grid search the batch size, epochs, dropout_rate and learning_rate

In [ ]:
# Import libraries
# If needed, run in a separate cell in your Puhti notebook environment:
# %pip install "tensorflow>=2.16,<2.18" "scikeras==0.13.0" "scikit-learn==1.5.2" pandas
# Restart the kernel after installation, then run all cells from the top.
from pathlib import Path
from time import perf_counter
import os

# One search worker avoids multiple TensorFlow processes competing for memory.
# On Puhti, use only the CPU threads allocated to your job.
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", os.environ.get("SLURM_CPUS_PER_TASK", "1"))
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasClassifier
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display

SEED = 42
SEARCH_JOBS = 1
keras.utils.set_random_seed(SEED)


In [ ]:
# Dataset splitting
# Place your local CSV beside this notebook, or change this path.
DATA_PATH = Path("pima-indians-diabetes.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}. Set DATA_PATH to your local CSV.")

# Accept both the tutorial's headerless CSV and diabetes.csv with column names.
data = pd.read_csv(DATA_PATH, header=None)
first_row = pd.to_numeric(data.iloc[0], errors="coerce")
if first_row.isna().all():
    data = data.iloc[1:].reset_index(drop=True)
data = data.apply(pd.to_numeric, errors="raise")
if data.shape[1] != 9:
    raise ValueError("Expected 9 columns: 8 features followed by the binary outcome.")
if data.iloc[:, -1].isna().any() or not data.iloc[:, -1].isin([0, 1]).all():
    raise ValueError("The last column must contain binary labels 0 and 1.")

features = data.iloc[:, :8]..copy()
labels = data.iloc[:, 8].astype("int32")
# Zero denotes missing measurements in these five Pima feature columns.
# Pregnancy count and the other columns retain their zero values.
features.iloc[:, [1, 2, 3, 4, 5]] = features.iloc[:, [1, 2, 3, 4, 5]].replace(0, np.nan)

# X and Y are only the training data; the final test set stays outside both searches.
X, X_test, Y, Y_test = train_test_split(
    features.to_numpy(), labels.to_numpy(), test_size=0.20,
    stratify=labels, random_state=SEED
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
print(f"Training samples: {len(Y)}; held-out test samples: {len(Y_test)}")

# Define the grid search parameters as follows:
# learning_rate:[0.001, 0.01, 0.1]
# batch_size: [10, 40 , 80]
# number of epochs: [10, 50, 100]
# dropout_rate:  [0.0, 0.3, 0.5]
learning_rate = [0.001, 0.01, 0.1]
batch_size = [10, 40, 80]
epochs = [10, 50, 100]
dropout_rate = [0.0, 0.3, 0.5]

# NOTE: If you don't have enough memory in your computer, drops some parameters


In [ ]:
#Create model
def create_model(learning_rate=0.001, dropout_rate=0.0):
    keras.backend.clear_session()
    #Input layer = 8
    inputs = keras.Input(shape=(8,))
    #Only one hidden layer = 12 Dense nodes, activation function = relu
    hidden = keras.layers.Dense(12, activation="relu")(inputs)
    #Dropout layer
    hidden = keras.layers.Dropout(dropout_rate)(hidden)
    #Output layer: 1 node, activation = sigmoid
    outputs = keras.layers.Dense(1, activation="sigmoid")(hidden)
    network = keras.Model(inputs=inputs, outputs=outputs)
    network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    return network


#Define KerasClassifier
classifier = KerasClassifier(
    model=create_model, model__learning_rate=0.001, model__dropout_rate=0.0,
    random_state=SEED, verbose=0
)
# Each CV fold learns its own medians, means, and standard deviations.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("nn", classifier)
])


In [ ]:
#Set parameter dictionary
param_grid = dict(
    nn__model__learning_rate=learning_rate,
    nn__batch_size=batch_size,
    nn__epochs=epochs,
    nn__model__dropout_rate=dropout_rate
)
#Define grid = GridSearcCV(....)
grid = GridSearchCV(
    estimator=model, param_grid=param_grid, n_jobs=SEARCH_JOBS, cv=cv,
    scoring="accuracy", verbose=1, refit=True, error_score="raise"
)
print(f"Grid search: {len(learning_rate) * len(batch_size) * len(epochs) * len(dropout_rate)} candidates, 3 folds each.")


In [ ]:
#Results
grid_start = perf_counter()
grid_result = grid.fit(X, Y)
grid_seconds = perf_counter() - grid_start


In [ ]:
#Summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))



## Random Search

In [ ]:
# Import libraries
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

#Create model
#Same as in Grid search example 
# Reuse create_model through a fresh clone of the same preprocessing/model pipeline.

#Define KerasClassifier
random_model = clone(model)


In [ ]:
# define the Random search parameters
# randomly from same range as the Grid search

#Set parameter dictionary
param_distributions = param_grid.copy()
N_ITER = 20

#Define random_search = RandomizedSearchCV(...)
random_search = RandomizedSearchCV(
    estimator=random_model, param_distributions=param_distributions,
    n_iter=N_ITER, scoring="accuracy", n_jobs=SEARCH_JOBS, cv=cv,
    random_state=SEED, verbose=1, refit=True, error_score="raise"
)
print(f"Random search: {N_ITER} sampled candidates, 3 folds each.")


In [ ]:
#Results
random_start = perf_counter()
random_search.fit(X, Y)
random_seconds = perf_counter() - random_start


In [ ]:
#Summarize results
print("Best: %f using %s" % (random_search.best_score_, random_search.best_params_))
means = random_search.cv_results_['mean_test_score']
stds = random_search.cv_results_['std_test_score']
params = random_search.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))



# Conclusion

__NOTE: Conclusion is one evaluation criterion for the exercise, so pay attention to writing this__

Write conclusion:




In [ ]:
# Generate the conclusion from actual results after both searches finish.
comparison = pd.DataFrame([
    {"Search": "Grid", "Candidates": len(grid_result.cv_results_["params"]),
     "Best CV accuracy": grid_result.best_score_,
     "CV standard deviation": grid_result.cv_results_["std_test_score"][grid_result.best_index_],
     "Elapsed seconds": grid_seconds},
    {"Search": "Random", "Candidates": len(random_search.cv_results_["params"]),
     "Best CV accuracy": random_search.best_score_,
     "CV standard deviation": random_search.cv_results_["std_test_score"][random_search.best_index_],
     "Elapsed seconds": random_seconds}
])
display(comparison)

# Choose using CV alone, then evaluate only that selected model on the held-out set.
selected_name, selected_search = max(
    [("Grid", grid_result), ("Random", random_search)],
    key=lambda item: item[1].best_score_
)
test_predictions = selected_search.predict(X_test)
test_accuracy = accuracy_score(Y_test, test_predictions)
print(f"Selected search: {selected_name}")
print(f"Held-out test accuracy: {test_accuracy:.4f}")
print("Confusion matrix (rows = true, columns = predicted; labels 0, 1):")
print(confusion_matrix(Y_test, test_predictions, labels=[0, 1]))
print(classification_report(Y_test, test_predictions, labels=[0, 1], zero_division=0))

display(Markdown(
    f"Grid search evaluated **{len(grid_result.cv_results_['params'])} combinations** "
    f"and achieved a best mean validation accuracy of **{grid_result.best_score_:.2%}**, "
    f"taking **{grid_seconds / 60:.2f} minutes**. "
    f"Random search sampled **{len(random_search.cv_results_['params'])} combinations** "
    f"and achieved **{random_search.best_score_:.2%}**, "
    f"taking **{random_seconds / 60:.2f} minutes**. "
    "Both searches used the same three stratified folds, architecture, seed, and preprocessing. "
    "Random search trades coverage for a smaller training budget; it may miss the best grid combination. "
    "Because it samples a subset of this exact grid, its best CV score should not exceed the full grid's "
    "when training is reproducible.\n\n"
    f"Using CV accuracy to select the search, **{selected_name} search** was selected "
    f"with parameters `{selected_search.best_params_}`. "
    f"Its refitted model achieved **{test_accuracy:.2%} accuracy** on the untouched test set. "
    "Median imputation and standardization were fitted only on each training fold, preventing "
    "preprocessing leakage. The test set was excluded from hyperparameter selection.\n\n"
    "These results come from one split and one training seed; fold variability, "
    "the small test set, and possible GPU nondeterminism limit the precision of the comparison. "
    "Accuracy alone does not describe errors for each class, so the classification report "
    "and confusion matrix should also be reviewed. Further tuning should use training CV, "
    "while preserving an independent final evaluation set."
))
